# Phase 4 — MobileNetV2 transfer learning

Enable **T4 GPU** before running. This notebook trains the project’s main model and compares fairly with the Custom CNN.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from shutil import copy2
from tqdm.auto import tqdm
import json, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf

PROJECT_ROOT = Path('/content/drive/MyDrive/waste-classification')
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'phase2_splits'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'phase4_mobilenetv2'
MODEL_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)
TARGET_CLASSES = ('plastic', 'paper', 'glass', 'metal', 'organic')
CLASS_TO_INDEX = {name: i for i, name in enumerate(TARGET_CLASSES)}
IMAGE_SIZE, BATCH_SIZE, RANDOM_SEED = (224, 224), 32, 42
tf.keras.utils.set_random_seed(RANDOM_SEED)
assert tf.config.list_physical_devices('GPU'), 'Select T4 GPU in Runtime settings, then reconnect.'
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
train_df = pd.read_csv(SPLIT_DIR / 'train.csv')
val_df = pd.read_csv(SPLIT_DIR / 'validation.csv')
test_df = pd.read_csv(SPLIT_DIR / 'test.csv')
with open(SPLIT_DIR / 'configuration.json') as file: config = json.load(file)
class_weights = {int(key): value for key, value in config['class_weights'].items()}

# Copy once to the fast temporary Colab disk; the Drive originals remain unchanged.
drive_root = PROJECT_ROOT / 'data' / 'raw' / 'mwcd'
local_root = Path('/content/waste-classification-local-data')
if not local_root.exists():
    images = [p for p in drive_root.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}]
    for source in tqdm(images, desc='Copying images to local disk'):
        destination = local_root / source.relative_to(drive_root)
        destination.parent.mkdir(parents=True, exist_ok=True)
        copy2(source, destination)

def local_path(path): return str(local_root / Path(path).relative_to(drive_root))
for frame in (train_df, val_df, test_df): frame['path'] = frame['path'].map(local_path)
print(f'Train {len(train_df):,} | Validation {len(val_df):,} | Test {len(test_df):,}')

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    return tf.cast(tf.image.resize(image, IMAGE_SIZE), tf.float32), label
def make_dataset(frame, training=False):
    paths = frame['path'].to_numpy(); labels = frame['label'].map(CLASS_TO_INDEX).to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training: dataset = dataset.shuffle(len(frame), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    return dataset.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
train_ds, val_ds, test_ds = make_dataset(train_df, True), make_dataset(val_df), make_dataset(test_df)

augment = tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'), tf.keras.layers.RandomRotation(.10), tf.keras.layers.RandomZoom(.10)], name='augmentation')
base = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3))
base.trainable = False
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
x = augment(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(.30)(x)
outputs = tf.keras.layers.Dense(5, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs, name='mobilenetv2_waste_classifier')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
model_path = MODEL_DIR / 'mobilenetv2_best.keras'
def callbacks():
    return [tf.keras.callbacks.ModelCheckpoint(model_path, monitor='val_accuracy', mode='max', save_best_only=True, verbose=1), tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1), tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=.5, min_lr=1e-6, verbose=1)]

# Stage 1: train only the new classification head.
head_history = model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weights, callbacks=callbacks())

# Stage 2: fine-tune the last 30 MobileNetV2 layers with a very small learning rate.
base.trainable = True
for layer in base.layers[:-30]: layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
fine_history = model.fit(train_ds, validation_data=val_ds, initial_epoch=len(head_history.history['loss']), epochs=len(head_history.history['loss']) + 8, class_weight=class_weights, callbacks=callbacks())

In [ ]:
history = {key: head_history.history[key] + fine_history.history[key] for key in head_history.history}
pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].plot(history['accuracy'], label='Train'); axes[0].plot(history['val_accuracy'], label='Validation'); axes[0].set_title('MobileNetV2 accuracy'); axes[0].legend()
axes[1].plot(history['loss'], label='Train'); axes[1].plot(history['val_loss'], label='Validation'); axes[1].set_title('MobileNetV2 loss'); axes[1].legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160); plt.show()
best_model = tf.keras.models.load_model(model_path)
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=1)
with open(OUTPUT_DIR / 'test_results.json', 'w') as file: json.dump({'test_loss':float(test_loss),'test_accuracy':float(test_accuracy)}, file, indent=2)
print(f'MobileNetV2 test accuracy: {test_accuracy:.2%}')
print('Best model:', model_path)